In [1]:
import mlflow

mlflow.set_experiment("Credit Card Fraud Detection")

print("MLflow experiment configured.")


import joblib

model = joblib.load("creditCardFraud/models/final_xgb_model.joblib")

print("Model type:", type(model).__name__)
print("Is fitted:", hasattr(model, "booster_"))
print("Number of estimators:", model.n_estimators)



MLflow experiment configured.
Model type: XGBClassifier
Is fitted: False
Number of estimators: 200


In [2]:
mlflow.set_experiment("Credit Card Fraud Detection")

print("Experiment ready.")

Experiment ready.


In [3]:
with mlflow.start_run():
    print("MLflow run started.")

MLflow run started.


In [4]:
with mlflow.start_run():
    mlflow.log_params({
        "model":"XGBoost",
        "n_estimators":200,
        "max_depth":6,
        "learning_rate":0.1,
        "subsample":0.8,
        "colsample_bytree":0.8,
        "scale_pos_weight":599.4761904761905,
    })

    print("Paramaters logged")

Paramaters logged


In [5]:
with mlflow.start_run():
    mlflow.log_metrics({
        "precision":0.9048,
        "recall":0.8000,
        "f1_score":0.8492,
        "roc_auc":0.9761,
        "pr_auc":0.8248,
    })

    print("Metrics logged. ")

Metrics logged. 


In [6]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)



In [7]:
X_train = pd.read_csv("creditCardFraud/data/processed/X_train_scaled.csv")
y_train = pd.read_csv("creditCardFraud/data/processed/y_train.csv").squeeze("columns")

X_test = pd.read_csv("creditCardFraud/data/processed/X_test_scaled.csv")
y_test = pd.read_csv("creditCardFraud/data/processed/y_test.csv").squeeze("columns")

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(226980, 30) (226980,)
(56746, 30) (56746,)


In [8]:
scale_pos_weight = (y_train == 0).sum()/(y_train == 1).sum()

model = XGBClassifier(
    n_estimators = 200,
    max_depth = 6,
    learning_rate = 0.1,
    subsample = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = scale_pos_weight,
    objective = "binary:logistic",
    eval_metric = "logloss",
    random_state = 42,
    n_jobs = -1,
)

print(f"scale_pos_weight: {scale_pos_weight}")

scale_pos_weight: 599.4761904761905


In [12]:

from sklearn.metrics import average_precision_score, roc_auc_score

current_proba = model.predict_proba(X_test)[:, 1]

print("current PR-AUC:", average_precision_score(y_test, current_proba))
print("Current ROC-AUC:", roc_auc_score(y_test, current_proba))
print("Current probability range:", current_proba.min(), current_proba.max())

current PR-AUC: 0.8248384759168829
Current ROC-AUC: 0.9761103301934559
Current probability range: 4.4280807e-08 0.9999987


In [13]:
with mlflow.start_run(run_name = "Final XGBoost - verified") as run:

    y_proba = model.predict_proba(X_test)[:, 1]

    threshold = 0.2325
    y_pred = (y_proba >= threshold).astype(int)

    metrics = {
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    mlflow.log_params({
        "model": "XGBoost",
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight":scale_pos_weight,
        "threshold":threshold,
    })

    mlflow.log_metrics(metrics)

    mlflow.xgboost.log_model(
        model,
        name = "xgboost_model"
    )


    print("Run ID:", run.info.run_id)
    print(metrics)


Run ID: 7ca4cb007278466ab4913ec4127cff24
{'precision': 0.9047619047619048, 'recall': 0.8, 'f1_score': 0.8491620111731844, 'roc_auc': 0.9761103301934559, 'pr_auc': 0.8248384759168829}


In [14]:
runs = mlflow.search_runs()

runs[[
"run_id",
"metrics.pr_auc",
"metrics.recall",
"metrics.precision",
"metrics.f1_score",
"params.model",
"params.threshold",
]]


,run_id,metrics.pr_auc,metrics.recall,metrics.precision,metrics.f1_score,params.model,params.threshold
0,7ca4cb007278466ab4913ec4127cff24,0.824838,0.8,0.904762,0.849162,XGBoost,0.2325
1,dac6477cd7b545aa8c4029569d001930,0.824800,0.8,0.904800,0.849200,None,None
2,134958676e1746d9a321cc84f8328478,NaN,NaN,NaN,NaN,XGBoost,None
3,41fb6331d59446718380c6601da21f65,NaN,NaN,NaN,NaN,None,None
4,ee0706d8c74348529b95be2254b47332,0.824838,0.8,0.904762,0.849162,XGBoost,0.2325
5,9abcf7e40d5144c1b0ceb55c427b279b,0.824800,0.8,0.904800,0.849200,None,None
6,8c8b6a25de7d4ff58901672ba057b261,NaN,NaN,NaN,NaN,XGBoost,None
7,b06506f19d5745e5831a6c20ba659f82,NaN,NaN,NaN,NaN,None,None
8,06554cff9a674db98af8c81167fa4752,0.824838,0.8,0.904762,0.849162,XGBoost,0.2325
9,d6ccbf1404f143f6b573f9af1f8e0d9e,0.824800,0.8,0.904800,0.849200,None,None


In [15]:
import os
print("Notebook directory:", os.getcwd())
print("MLflow tracking URI:", mlflow.get_tracking_uri())

Notebook directory: C:\Users\hnkru
MLflow tracking URI: sqlite:///C:/Users/hnkru/mlflow.db


In [10]:
import joblib

MODEL_PATH = "creditCardFraud/models/final_xgb_model.joblib"

model = joblib.load(MODEL_PATH)

# If the saved artifact is unfitted, retrain it
if not hasattr(model, "booster_"):
    model.fit(X_train, y_train)

from sklearn.metrics import average_precision_score, roc_auc_score

current_proba = model.predict_proba(X_test)[:, 1]

print("current PR-AUC:", average_precision_score(y_test, current_proba))
print("Current ROC-AUC:", roc_auc_score(y_test, current_proba))
print("Current probability range:", current_proba.min(), current_proba.max())

current PR-AUC: 0.8248384759168829
Current ROC-AUC: 0.9761103301934559
Current probability range: 4.4280807e-08 0.9999987


In [16]:
import mlflow

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())

client = mlflow.MlflowClient()

run = client.get_run("ee0706d8c74348529b95be2254b47332")

print("Run status:", run.info.status)
print("Run ID:", run.info.run_id)
print("Artifact URI:", run.info.artifact_uri)

MLflow version: 3.16.0
Tracking URI: sqlite:///C:/Users/hnkru/mlflow.db
Run status: FINISHED
Run ID: ee0706d8c74348529b95be2254b47332
Artifact URI: file:///C:/Users/hnkru/mlruns/2/ee0706d8c74348529b95be2254b47332/artifacts


In [17]:
logged_models = mlflow.search_logged_models(
    filter_string=f"source_run_id = '{run.info.run_id}'"
)

logged_models

,artifact_location,creation_timestamp,experiment_id,last_updated_timestamp,metrics,model_id,model_type,name,params,source_run_id,status,status_message,tags
0,file:///C:/Users/hnkru/mlruns/2/models/m-61ebd...,1789741649386,2,1789741687156,"[<Metric: dataset_digest=None, dataset_name=No...",m-61ebd9aa2a1b4b608e8e14e20f93caed,None,xgboost_model,"{'colsample_bytree': '0.8', 'learning_rate': '...",ee0706d8c74348529b95be2254b47332,READY,None,"{'mlflow.modelVersions': '[{""name"": ""CreditCar..."


In [18]:
model_id = "m-61ebd9aa2a1b4b608e8e14e20f93caed"

registered_model = mlflow.register_model(
    model_uri=f"models:/{model_id}",
    name="CreditCardFraud-XGBoost"
)

print("Model name:", registered_model.name)
print("Model version:", registered_model.version)

Model name: CreditCardFraud-XGBoost
Model version: 2


Registered model 'CreditCardFraud-XGBoost' already exists. Creating a new version of this model...
Created version '2' of model 'CreditCardFraud-XGBoost'.


In [20]:
client = mlflow.MlflowClient()

versions = client.search_model_versions(
    filter_string="name='CreditCardFraud-XGBoost'"
)

for v in versions:
    print(
        "Name:", v.name,
        "| Version:", v.version,
        "| Run ID:", v.run_id
    )

Name: CreditCardFraud-XGBoost | Version: 2 | Run ID: ee0706d8c74348529b95be2254b47332
Name: CreditCardFraud-XGBoost | Version: 1 | Run ID: ee0706d8c74348529b95be2254b47332


In [21]:
run_v2 = client.get_run("c04022290d2e49899579f565a9d4e667")

print("Run status:", run_v2.info.status)
print("Run ID:", run_v2.info.run_id)

Run status: FINISHED
Run ID: c04022290d2e49899579f565a9d4e667


In [22]:
model_v2 = mlflow.search_logged_models(
    filter_string="source_run_id = 'c04022290d2e49899579f565a9d4e667'"
)

model_v2

,artifact_location,creation_timestamp,experiment_id,last_updated_timestamp,metrics,model_id,model_type,name,params,source_run_id,status,status_message,tags
0,file:///C:/Users/hnkru/mlruns/2/models/m-75f41...,1789496524301,2,1789496543025,"[<Metric: dataset_digest=None, dataset_name=No...",m-75f415d205b24532a56b3a9f5f2181db,None,xgboost_model,"{'colsample_bytree': '0.8', 'learning_rate': '...",c04022290d2e49899579f565a9d4e667,READY,None,{'mlflow.source.name': '07_experiment_tracking...


In [23]:
model_v2_id = model_v2.iloc[0]["model_id"]

registered_v2 = mlflow.register_model(
    model_uri=f"models:/{model_v2_id}",
    name="CreditCardFraud-XGBoost"
)

print("Model name:", registered_v2.name)
print("Model version:", registered_v2.version)

Model name: CreditCardFraud-XGBoost
Model version: 3


Registered model 'CreditCardFraud-XGBoost' already exists. Creating a new version of this model...
Created version '3' of model 'CreditCardFraud-XGBoost'.


In [24]:
versions = client.search_model_versions(
    filter_string="name='CreditCardFraud-XGBoost'"
)

for v in versions:
    print(
        f"Version: {v.version} | "
        f"Run ID: {v.run_id} | "
        f"Status: {v.status}"
    )

Version: 3 | Run ID: c04022290d2e49899579f565a9d4e667 | Status: READY
Version: 2 | Run ID: ee0706d8c74348529b95be2254b47332 | Status: READY
Version: 1 | Run ID: ee0706d8c74348529b95be2254b47332 | Status: READY


In [25]:
v3 = client.get_model_version(
    name="CreditCardFraud-XGBoost",
    version="3"
)

print("Version:", v3.version)
print("Run ID:", v3.run_id)
print("Source:", v3.source)

Version: 3
Run ID: c04022290d2e49899579f565a9d4e667
Source: models:/m-75f415d205b24532a56b3a9f5f2181db


In [26]:
client.set_model_version_tag(
    name="CreditCardFraud-XGBoost",
    version="3",
    key="validation_status",
    value="validated"
)

client.set_model_version_tag(
    name="CreditCardFraud-XGBoost",
    version="3",
    key="pr_auc",
    value="0.8248"
)

client.set_model_version_tag(
    name="CreditCardFraud-XGBoost",
    version="3",
    key="threshold",
    value="0.2325"
)

print("Version 3 metadata added successfully.")

Version 3 metadata added successfully.


In [27]:
client.set_registered_model_alias(
    name="CreditCardFraud-XGBoost",
    alias="candidate",
    version="3"
)

print("Version 3 assigned alias: candidate")

Version 3 assigned alias: candidate


In [28]:
candidate = client.get_model_version_by_alias(
    name="CreditCardFraud-XGBoost",
    alias="candidate"
)

print("Candidate version:", candidate.version)
print("Run ID:", candidate.run_id)

Candidate version: 3
Run ID: c04022290d2e49899579f565a9d4e667


In [29]:
client.set_registered_model_alias(
    name="CreditCardFraud-XGBoost",
    alias="production",
    version="3"
)

print("Version 3 assigned alias: production")

Version 3 assigned alias: production


In [30]:
candidate = client.get_model_version_by_alias(
    name="CreditCardFraud-XGBoost",
    alias="candidate"
)

production = client.get_model_version_by_alias(
    name="CreditCardFraud-XGBoost",
    alias="production"
)

print("Candidate:", candidate.version)
print("Production:", production.version)

Candidate: 3
Production: 3


In [31]:
production_model = mlflow.pyfunc.load_model(
    "models:/CreditCardFraud-XGBoost@production"
)

print("Production model loaded successfully.")
print(type(production_model))

Production model loaded successfully.
<class 'mlflow.pyfunc.PyFuncModel'>


In [32]:
production_xgb = mlflow.xgboost.load_model(
    "models:/CreditCardFraud-XGBoost@production"
)

production_proba = production_xgb.predict_proba(X_test)[:, 1]

print("Production PR-AUC:",
      average_precision_score(y_test, production_proba))

print("Production ROC-AUC:",
      roc_auc_score(y_test, production_proba))

Production PR-AUC: 0.8248384759168829
Production ROC-AUC: 0.9761103301934559


In [33]:
client.set_registered_model_alias(
    name="CreditCardFraud-XGBoost",
    alias="production",
    version="2"
)

production = client.get_model_version_by_alias(
    name="CreditCardFraud-XGBoost",
    alias="production"
)

print("Production version after rollback:", production.version)

Production version after rollback: 2


In [34]:
client.set_registered_model_alias(
    name="CreditCardFraud-XGBoost",
    alias="production",
    version="3"
)

production = client.get_model_version_by_alias(
    name="CreditCardFraud-XGBoost",
    alias="production"
)

print("Final production version:", production.version)

Final production version: 3
